# TP4 - Sampling and Bootstrapping


Estimated time: **1.5 h** 


### 🎯 Learning Objectives

In this lab, you'll go over hands-on exercises using Python to further consolidate your understanding about sampling bias and diverse sampling techniques, sampling distribution, the Central Limit Theorem, as well as standard error. Besides, you will explore how to use bootstrap algorithm to find the confidence interval for sample mean. 

### 🍄 Outline


* Get familiar with the dataset
* Sample exercises
  * Sampling bias and sampling techniques (random sampling, stratified sampling, etc)
  * Sampling distribution, Central Limit Theorem (CLT), standard error
  * Bootstrap for confidence interval
* Assignment questions

### 📚 Import Libraries


Install required libraries for this lab. `pip` will automatically skip already installed packages

In [ ]:
import importlib.util

# import name: pip installation name
required_pkgs = {
    "numpy": "numpy",
    "pandas": "pandas",
    "matplotlib": "matplotlib",
    "seaborn": "seaborn",
    "sklearn": "scikit-learn",
    "statsmodels": "statsmodels",
    "scipy": "scipy",
}

missing_pkgs = [
    pip_name
    for import_name, pip_name in required_pkgs.items()
    if importlib.util.find_spec(import_name) is None
]

if missing_pkgs:
    print("⚠️ Missing packages:", ", ".join(missing_pkgs))
    packages = " ".join(missing_pkgs)
    %pip install $packages
else:
    print("🎉 All packages already installed.")

Import the required packages


In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.utils import resample

## Get familiar with the dataset

For this TP we will make use of the `loans_income.csv` data set. It contains the annual income for 50000 loan applicants.

#### Load and display the dataset

In [ ]:
df = pd.read_csv('loans_income.csv')
df

#### Load the dataset, and convert it into a Pandas Series since it contains only 1 column of data

In [ ]:
# Load the 'loans_income.csv' file, and squeeze() ensures that if the CSV has a single column, it becomes a Series instead of a DataFrame
loans_income = pd.read_csv('loans_income.csv').squeeze('columns')

# Display the loaded income data
loans_income

#### Produce a descriptive statistics table

In [ ]:
loans_income.describe().round(2)

#### Generate the histogram corresponding to the dataset

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(loans_income, bins=20, kde=True)
plt.title('Annual Income Distribution for 50000 loan applicants')
plt.xlabel('Income')
plt.ylabel('Count')

plt.show()

# 📝 Example exercise

## Part 1: Sampling bias and sampling techniques (simple random sampling, proportional stratified sampling, equal stratified sampling) 

### <span style="color:blue"> Example 1: Illustration of sampling bias </span>

**<span style="color:blue">Purpose</span>:** demonstrate how a biased sampling mechanism can distort estimates, and showcase the importance of sample representativeness.

**<span style="color:blue">Task</span>:** We'll construct a biased sample by selecting only from the top 30% incomes (as if we had access only to a specific sub-population). We'll compare the mean from this biased sample against an unbiased simple random sample of the same size,  as well as against the population mean.

**Step-1: Let's treat the whole dataset as our population, and compute the mean income of all loan applicants**

In [ ]:
pop_mean = loans_income.mean()
print(f"Population mean: {pop_mean:,.2f}")

**Step-2: Construct a biased sample by selecting only from the top 30% incomes (as if we had access only to a specific sub-population). Then take 1000 random samples from it and compute the sample mean**

In [ ]:
# Define 'biased frame': top 30% incomes
cut = np.quantile(loans_income, 0.7) # find the income value such that 70% of incomes are lower, 30% are higher
high_income = loans_income[loans_income >= cut] # keeps only the highest 30% of the incomes
print("The number of applicants with high income is:", len(high_income))

In [ ]:
# Take a random sample of 1000 observations without replacement from the biased frame
biased_sample = high_income.sample(n=1000,             # sample size
                                    replace=False,     # sampling without replacement
                                    random_state=2025  # random seed for reproducibility
                                   )
mean_biased = biased_sample.mean()

print(f"Biased sample mean (top 30% only): {mean_biased:,.2f}")

**Step-3: Conduct a random sampling from the whole population, and compute the sample mean**

In [ ]:
# Take a random sample of 1000 observations without replacement from the whole population.
unbiased_sample = loans_income.sample( n=1000,              # sample size
                                       replace=False,      # sampling without replacement 
                                       random_state=2025   # random seed for reproducibility
                                     )
mean_unbiased = unbiased_sample.mean()
print(f"Unbiased simple random sample mean: {mean_unbiased:,.2f}")

**Step-4: Compare the results**

----

❓<span style="color:purple;">**Quick test**</span>:  Given that the population mean is approximately **68,760.52**, which observation is best supported by the results from Steps 2 and 3?

- **A.** The two means differ only because different random observations were selected.
- **B.** Both samples are unbiased because observations were selected randomly in both cases.  
- **C.** Both samples are equally representative because they have the same sample size.
- **D.** The high-income sample is biased and leads to an overestimate of the population mean.

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;">D. The high-income sample is biased and leads to an overestimate of the population mean.</p>

**Explanation:**

<p style="color:blue;">Random selection within the high-income group does not remove the sampling bias because the sample itself is <b>not representative of the population</b>. By systematically excluding low- and middle-income applicants, the sampling procedure produces a mean that tends to overestimate the population mean. Having the same sample size or selecting randomly within a biased frame does not correct this problem.</p>

</details>

----

----

❓<span style="color:purple;">**Quick test**</span>:
If we take a very large biased sample (e.g., 5000 instead of 1000 but still from the high-income applicants), will the bias disappear? 

<details> <summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"> No — Larger size reduces variability in random sampling, but not systematic bias. </p>

**Explanation:**

<p style="color:blue;"> Increasing the sample size reduces <i>random variability</i> due to sampling. But if the sample itself is biased (e.g., excluding low- and middle-income applicants), then the sample mean will still systematically differ from the true population mean. This type of error is called <b>systematic bias</b>, and it cannot be fixed by increasing the sample size. </p> </details>

----

### <span style="color:blue"> Example 2: Preventing sampling bias with strata -- proportional stratified sampling </span>

**<span style="color:blue">Purpose</span>:** In this exercise, we will show how **stratified sampling help reduce sampling bias by ensuring better representativeness across strata**, in particular with a focus on proportional stratified sampling.

**<span style="color:blue">Task</span>:** We'll create income-based strata from the dataset `loans_income`. Then we will draw two samples of the same size (1000): one using **proportional stratified sampling** and the other using **simple random sampling**. After that, we'll compare the estimated means based on the two samples, and also against the population mean.


**Notes:** Since the dataset has only one column (annual income), we will define three income groups: **low** if annual income is less than `<50,000`, **middle** if annual income is within the range `[50000, 100000]`, and **high** if annual income `>100,000`. **Each represents a stratum.**

**First of all, we will count the number of loan applicants in each group, as well as the proportion each group represents.**

In [ ]:
# Define three income groups
def income_status(income):
    if income < 50000:
        return "Low (<50k)"
    elif income <= 100000:
        return "Middle (50k–100k)"
    else:
        return "High (>100k)"

# Apply the classification to the dataset
df_income = pd.DataFrame({"income": loans_income})
df_income["income_status"] = df_income["income"].apply(income_status)

# Count number of loan applicants in each income group
income_counts = df_income["income_status"].value_counts()

# Convert counts into a DataFrame for better visualization
income_table = income_counts.reset_index()
income_table.columns = ["Income Group", "Count"]

# Add a column which indicates the proportion of loan applicants in each group
# .round(2) keeps 2 digits after the decimal point
total = income_table["Count"].sum()
income_table["Percentage"] = (income_table["Count"] / total).round(2)

income_table

----

❓<span style="color:purple;">**Quick test**</span>:  Suppose we want a total sample of **1,000 applicants**. Under proportional stratified sampling, how many observations should be selected from each income group?

- **A.** Middle: 333, Low: 333, High: 334  
- **B.** Middle: 540, Low: 310, High: 150 
- **C.** Middle: 150, Low: 310, High: 540  

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"><b>B. Middle: 540, Low: 310, High: 150.</b></p>

**Explanation:**

<p style="color:blue;">Under proportional stratified sampling, the number selected from each group reflects its population share. Therefore:</p>

<p style="color:blue;">
Middle: \(1000 \times 0.54 = 540\)<br>
Low: \(1000 \times 0.31 = 310\)<br>
High: \(1000 \times 0.15 = 150\)
</p>

<p style="color:blue;">These allocations sum to the required total sample size of 1,000.</p>

</details>

----

**Display the dataset which now indicates also the income group each loan applicant belongs to**

In [ ]:
df_income

#### **Example 2-1: Conduct the proportional stratified sampling and estimate the mean income**

First of all, we need to determine the number of samples to be taken from each stratum (income group). In total, we need 1000 samples. **For proportional stratified sampling, each stratum will be sampled in proportion to its size**. For instance, loan applicants with Middle income represents 54% of the population. Proportionally, 1000*0.54=540 samples will be taken from the Middle stratum.

In [ ]:
n_total = 1000  # desired total sample size

# Add a column to indicate how many samples to be taken from each stratum
income_table["Sample size (Proportional)"] = (income_table["Percentage"]*n_total).astype(int)
income_table

The last column `Sample size (Proportional)`indicates the number of samples to be taken from each stratum, which we will use in the following to perform the proportional stratified sampling. **Within each stratum, selection is random**.

In [ ]:
# Use to store the samples taken from each stratum
# for proportional stratified sampling.
samples_prop = []

# Sample within each stratum (without replacement)
for g in df_income["income_status"].unique():
    # extract the stratum
    pool = df_income[df_income["income_status"] == g]
    # determine the number of samples to be taken from the stratum
    sample_size_per_group = int(income_table[income_table["Income Group"]==g]["Sample size (Proportional)"].iloc[0])
    # take the desired number of samples from the stratum, and append the results
    # replace=False: sample without replacement. random_state: random number used to ensure result reproducibility
    samples_from_stratum = pool.sample(n=sample_size_per_group, replace=False, random_state=2025)["income"]
    samples_prop.append(samples_from_stratum)

# Combines the samples drawn from the 3 strata into a single sample
sample_prop = pd.concat(samples_prop, ignore_index=True)

# Compute the sample mean and the absolution 
# error w.r.t to the population mean
mean_prop = sample_prop.mean()
err_prop = abs(mean_prop - pop_mean)

print(f"Population mean: {pop_mean:,.2f}")
print(f"Proportional stratified sample mean: {mean_prop:,.2f}")
print(f"Absolute error (proportional): {err_prop:,.2f}")

Besides the sample mean, above code also computes the **absolute error between the sample mean and the population mean**, regardless of direction (positive or negative). We use it as a simple way to compare different sampling methods. As we can see, proportional stratified sampling obtains a sample mean which is 776.33 away from the true population mean. 

#### **Example 2-2: Conduct a simple random sampling, and compute the sample mean**

For comparison purpose, we also perform a simple random sampling, **each data point has an equal chance of being selected**. As for proportional stratified sampling, we **sample without replacement** from the dataset.

In [ ]:
# Take a random sample of 1000 observations without replacement from the dataset.
samples_simple_random = loans_income.sample(n=1000,                    # sample size
                                            replace=False,             # sampling without replacement 
                                            random_state=2025          # random seed for reproducibility
                                            )

# Compute the sample mean and the absolution 
# error w.r.t to the population mean
mean_random = samples_simple_random.mean()
err_random = abs(mean_random - pop_mean)

print(f"Population mean: {pop_mean:,.2f}")
print(f"Simple random sample mean: {mean_random:,.2f}")
print(f"Absolute error (simple random): {err_random:,.2f}")

#### **Example 2-3: Visualize the sample mean for different sampling techniques, and make observation**

Below we build a dataframe for the sample means and the absolution errors computed based on proportional stratified sampling and simple random sampling. In addition, we include also the population mean.

In [ ]:
summary_table = pd.DataFrame({
    "Mean": [
        pop_mean,
        mean_prop,
        mean_random
    ],
    "Absolute Error": [
        0.0,           # population mean has no error
        err_prop,
        err_random        
    ]
}, index=["Population", "Proportional Stratified", "Simple random"])

# Round for readability: keep 2 digits after 
# the decimal point.
summary_table = summary_table.round(2)
summary_table


Besides, we also generate a plot to visualize the sample means obtained with each technique.

In [ ]:
# --- Comparison Plot: Population vs Proportional vs Simple random ---
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 1.5))

# Population mean (truth)
plt.axvline(pop_mean, color="red", linestyle="--", linewidth=2, label="Population mean")

# Proportional stratified sample mean
plt.axvline(mean_prop, color="green", linestyle="-", linewidth=2, label="Stratified mean")

# Simple random sample mean
plt.axvline(mean_random, color="purple", linestyle="-", linewidth=2, label="Simple random mean")

plt.title("Comparison of Sampling Strategies")
plt.xlabel("Income")
plt.yticks([])  # hide y-axis for clarity
plt.legend()
plt.show()


As we can see, proportional stratified sampling yields estimate closer to the population mean than simple random sampling.
  

----

❓<span style="color:purple;">**Quick test**</span>: Based on the above results, can we conclude that proportional stratified sampling will **always** be better than simple random sampling at avoiding sampling bias?

- **A. Yes**
- **B. No**

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"><b>B. No.</b></p>

**Explanation:**

<p style="color:blue;">Both methods can produce unbiased estimates when implemented correctly. Proportional stratified sampling can improve subgroup representation and reduce sampling variability when the strata are relevant, but it does not guarantee an estimate closer to the population value in every sample. A single comparison is therefore insufficient to conclude that one method is always better.</p>

</details>

----

Below we generate a table which counts the number of observations **simple random sampling** selected from each group, as well as the percentage.

In [ ]:
# Assign each observation in the simple random sample to an income group
simple_random_groups = samples_simple_random.apply(income_status)

# Count the observations selected from each income group
group_counts = (
    simple_random_groups
    .value_counts()
    .reindex(income_table["Income Group"], fill_value=0)
)

# Create and display the summary table
simple_random_group_table = pd.DataFrame({
    "Income Group": group_counts.index,
    "Sample Count": group_counts.values,
    "Sample Percentage (%)": (
        group_counts.values / len(samples_simple_random) * 100
    ).round(2)
})

simple_random_group_table

----

❓<span style="color:purple;">**Quick test**</span>:  Change `random_state` in Examples 2-1 and 2-2 to `2026`, and then rerun the code cells in Example 2-1 to Example 2-3. What is the most appropriate observation?

- **A.** The population mean changes when the random seed changes.
- **B.** Proportional stratified sampling always remains closer to the population mean.
- **C.** The sample means and absolute errors change, and the method producing the closer estimate may differ between random samples.
- **D.** Changing the random seed introduces sampling bias into both methods.

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"><b>C. The sample means and absolute errors change, and the method producing the closer estimate may differ between random samples.</b></p>

**Explanation:**

<p style="color:blue;">Changing the random seed changes which observations are selected, so the sample estimates vary because of random sampling variability. The population mean is the mean of the whole population, it is not affected by sampling. Hence, it remains unchanged. Therefore, the result from one random sample is not sufficient to conclude that one method will always produce an estimate closer to the population mean. </p>

</details>

----

## Part 2: Sampling distribution and Central Limit Theorem

### <span style="color:blue"> Example 3: Plot the sampling distribution of sample mean. </span>

#### **Example 3-1: Plot the distribution of the population and the population mean**

In [ ]:
population_mean=loans_income.mean()

plt.figure(figsize=(8, 4))
# Generate a histogram corresponding to the population
sns.histplot(loans_income, bins=20, kde=False)

# Add a red vertical dashed-line corresponding to the population mean
plt.axvline(population_mean, color='red', linestyle='--', linewidth=2,
            label=f'Population mean: {population_mean:,.2f}')

plt.title('Population Distribution and Population Mean')
plt.xlabel('Income')
plt.ylabel('Count')
plt.legend()

plt.show()

According to the above plot, the population income distribution is **right-skewed**, with most applicants concentrated in the lower and middle income ranges and a relatively small number of applicants earning very high incomes. These high-income values create a long right tail and pull the population mean to approximately €68,760.52.

---

❓<span style="color:purple;">**Quick test**</span>: If we repeatedly draw random samples of 50 applicants from this right-skewed population and calculate the mean income of each sample, what should we expect?

- **A.** Every sample will have exactly the same mean as the population mean.
- **B.** The sample means will vary, but their distribution should be approximately bell-shaped (normal distribution).
- **C.** The sample means will have the same strongly right-skewed distribution as the individual incomes.
- **D.** The sample means will systematically be higher than the population mean.

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"><b>B. The sample means will vary, but their distribution should be approximately bell-shaped and centred near the population mean.</b></p>

**Explanation:**

<p style="color:blue;">Different random samples contain different applicants, so their means will vary because of random sampling variability. </p>
    
<p style="color:blue;">However, according to the Central Limit Theorem, for sufficiently large samples (i.e. here the sample size is 50), the sampling distribution of the mean is approximately normal.</p>

</details>

---

#### **Example 3-2: Plot the sampling distribution of sample mean with sample size of 50**

To examine the sampling distribution of the sample mean, below we repeatedly draw 1,000 random samples, each containing 50 applicants. We calculate the mean income of each sample and then plot the distribution of the resulting 1,000 sample means. 

**The mean of these sample means can be used to approximate the population mean**, while their spread illustrates random sampling variability.

In [ ]:
# Set the sample size and number of repeated samples
sample_size = 50
n_repetitions = 1000

# Set a random seed for reproducibility
rng = np.random.default_rng(seed=2026)

# Draw 1000 samples of size 50 and calculate each sample mean
sample_means_50 = []

for _ in range(n_repetitions):
    # Draw one random sample and calculate the sample mean
    one_sample = loans_income.sample(n=sample_size, replace=False, random_state=rng)
    sample_means_50.append(one_sample.mean())

# Convert the sample means into a Pandas Series
sample_means_50 = pd.Series(sample_means_50)

# Calculate the mean of the sample means
mean_of_sample_means = sample_means_50.mean()


plt.figure(figsize=(8, 4))
# Plot the sampling distribution of the sample mean
sns.histplot(sample_means_50, bins=20, kde=False)

# Add vertical lines for the population mean and the mean of sample means
plt.axvline(population_mean, color='red', linestyle='--',
            label=f'Population mean: {population_mean:,.2f}')
plt.axvline(mean_of_sample_means, color='green', linestyle='-',
            label=f'Mean of sample means: {mean_of_sample_means:,.2f}')

plt.title('Sampling Distribution of the Sample Mean (n = 50)')
plt.xlabel('Sample Mean Income')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

---

❓<span style="color:purple;">**Quick test**</span>: Based on the sampling distribution above, which of the following is a **wrong interpretation**?

- **A.** The approximately bell-shaped distribution is consistent with the Central Limit Theorem.
- **B.** The mean of the repeated sample means can be used to approximate the population mean.
- **C.** The spread of the sampling distribution reflects the random variability of sample means across repeated samples.
- **D.** Because the mean of the sample means is close to the population mean, every sample of 50 applicants will produce a mean close to the population mean.

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"><b>D. Because the mean of the sample means is close to the population mean, every sample of 50 applicants will produce a mean close to the population mean.</b></p>

**Explanation:**

<p style="color:blue;">The closeness of the two means suggests that <b>the average</b> of the sample means approximates the population mean. </p>
    
<p style="color:blue;">But it does not guarantee that every individual sample will produce a mean close to the population mean. As shown by the spread of the sampling distribution, individual sample means vary because of random sampling variability.</p>

</details>

---

#### **Example 3-3: Compute the Standard Error**

The Standard Error (SE) is the **standard deviation (SD) of the sampling distribution** of a statistic – it quantifies how much a sample statistic (e.g., sample mean) is expected to fluctuate from sample to sample due to random sampling variability.

In [ ]:
# Compute the standard error as the standard deviation
# of the sampling distribution of the sample mean
standard_error = sample_means_50.std()

print(f"Standard error: {standard_error:,.2f}")

The standard error indicates that, if we repeatedly take random samples of 50 applicants, their sample means typically vary by about €4,683.20 around the population mean.

#### **Example 3-4: Compare against the theoretical standard error**

The formula to calculate the theoretical standard error is:

$$
\large
\text{Standard Error} \quad  SE = \frac{\sigma}{\sqrt{n}}
$$

where, 
- $\sigma$ is the **standard deviation of the population**. If unknown, we can use the standard deviation of the sample;
- $n$ is the sample size, here it is **the number of observations used to compute the sample statistic**

In [ ]:
# Compute the theoretical standard error
theoretical_standard_error = loans_income.std() / np.sqrt(sample_size)

print(f"Empirical standard error: {standard_error:,.2f}")
print(f"Theoretical standard error: {theoretical_standard_error:,.2f}")

The **empirical standard error** estimates random sampling variability through simulation using repeated sampling, while the **theoretical standard error** calculates the expected variability mathematically. Their similarity indicates that the simulation is consistent with the theoretical formula.

---
❓<span style="color:purple;">**Quick test**</span>: If we **increase the sample size** (the number of applicants in each random sample), what would happen to the sampling distribution of the sample mean and its standard error?

- **A.** The sampling distribution becomes wider, and the standard error increases.
- **B.** The sampling distribution becomes narrower and more nearly normal, while the standard error decreases.
- **C.** The sampling distribution shifts away from the population mean, while the standard error remains unchanged.
- **D.** Neither the sampling distribution nor the standard error changes.

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"><b>B. The sampling distribution becomes narrower and more nearly normal, while the standard error decreases.</b></p>

**Explanation:**

<p style="color:blue;">A larger sample size reduces the random variability of the sample mean. The sample mean from sample to sample will fluctuate less, causing the sampling distribution to become narrower. According to the Central Limit Theorem, its shape also becomes more nearly normal as the sample size increases. </p>
    
<p style="color:blue;">Since \(SE=\frac{\sigma}{\sqrt{n}}\), increasing \(n\) decreases the standard error. For example, multiplying the sample size by four reduces the standard error by approximately half.</p>

</details>

---

#### **Example 3-5: Investigate the Central Limit Theorem by varying the sample size 50, 100, 200**

**<span style="color:blue">Purpose</span>:** demonstrate how increasing the sample size affects the shape and variability of the sampling distribution of the sample mean, illustrating the Central Limit Theorem and the relationship between sample size and standard error.

**<span style="color:blue">Task</span>:** We'll repeatedly draw 1,000 random samples for each sample size (50, 100, and 200) and calculate the mean income of each sample. We'll then plot and compare the resulting sampling distributions, and their empirical standard errors.

In [ ]:
# Set the sample sizes and number of repeated samples
sample_sizes = [50, 100, 200]
n_repetitions = 1000

# Create a random-number generator for reproducibility
rng = np.random.default_rng(seed=2026)

# Store the sample means obtained for different sample sizes
sampling_results = []

# Generate the sampling distribution for each sample size
for sample_size in sample_sizes:
    for _ in range(n_repetitions):
        # Draw one random sample from the population
        # and calculate the sample mean
        one_sample = loans_income.sample(n=sample_size, replace=False, random_state=rng)
        sample_mean = one_sample.mean()
        
        # Store the result
        sampling_results.append({
            "Sample Mean": sample_mean,
            "Sample Size": sample_size
        })

# Convert the results into a Pandas DataFrame
sampling_results = pd.DataFrame(sampling_results)

# Calculate the population mean
population_mean = loans_income.mean()

# Create one subplot row for each sample size, using a shared x-axis for comparison 
# while allowing each subplot to have its own y-axis scale
g = sns.FacetGrid(sampling_results, row="Sample Size", height=2.4, aspect=3, sharex=True, sharey=False)

# Plot the sampling distributions
g.map_dataframe(sns.histplot, x="Sample Mean", bins=25, kde=False)

# Add the population mean and empirical standard error to each plot
for ax, sample_size in zip(g.axes.flat, sample_sizes):
    # Extract the sample means for each sample size
    sample_means = sampling_results.loc[
        sampling_results["Sample Size"] == sample_size,
        "Sample Mean"
    ]
    # Compute the standard error based on the sampling distribution
    empirical_se = sample_means.std()

    # Add a vertical dashed-line for the population mean
    ax.axvline(population_mean, color="red", linestyle="--", linewidth=2,
               label=f"Population mean: {population_mean:,.2f}")
    # Add a textbox indicating the empirical standard error
    ax.text(0.98, 0.80, f"Empirical SE: {empirical_se:,.2f}",
            transform=ax.transAxes, horizontalalignment="right",
            color="blue")

    ax.legend()

g.set_titles("Sampling Distribution of the Sample Mean (n = {row_name})")
g.set_axis_labels("Sample Mean Income", "Count")

plt.tight_layout()
plt.show()

The above 3 subplots show the sampling distribution of the 1000 sample means with the sample size varying from 50, 100, 200. As we can see, **the tendency of the sampling distribution to approach a normal distribution as the sample size increases**.

Besides, **as the sample size increases, the standard error decreases**. We observe that, **to reduce the standard error of the sample mean by a factor of 2, the sample size needs to be increased by a factor of 4**. This can also be explained by the formula $SE = \frac{\sigma}{\sqrt{n}}$. 

As shown below, the standard error for sample size 200 is almost half of the standard error for sample size 50. From 50 to 200, the sample size increases by a factor of 4. 

In [ ]:
# Summarize the sampling distributions
standard_error_summary = (
    sampling_results
    .groupby("Sample Size")["Sample Mean"]  # Group sample means by sample size
    .agg(                                   
        Mean_of_sample_means="mean",        # Calculate the mean of the sample means for each sample size
        Empirical_SE="std"                  # Calculate the standard error for each sample size
    )
    .reset_index()
)

# Calculate the population standard deviation
population_std = loans_income.std(ddof=0)

# Calculate the theoretical standard error using SE = σ / sqrt(n)
standard_error_summary["Theoretical_SE"] = (
    population_std / np.sqrt(standard_error_summary["Sample Size"])
)

# Round the results for better readability
standard_error_summary = standard_error_summary.round(2)

standard_error_summary

## Part 3: Bootstrap for Confidence Interval

The bootstrap approximates the sampling distribution by **resampling with replacement from the observed sample**. It can be used to compute confidence intervals for many statistics, e.g. mean, median, etc. It does not create new information — it reuses the observed data to estimate variability.


<p align="right">
  <img src="bootstrap_algorithm.png" alt="Bootstrap algorithm" width="700"/>
</p>

<p align="right">
  <img src="bootstrap_algorithm-2.png" alt="Bootstrap algorithm -2" width="700"/>
</p>


### <span style="color:blue">  Example 4: Compute the 90% Confidence Interval (CI) of the sample mean using bootstraping, based on an initial sample of 20 data points from the loans income dataset. </span>

#### **Example 4-1: compute the 90% confidence interval for the sample mean**

In [ ]:
from sklearn.utils import resample

# Set a random seed for reproducibility
np.random.seed(seed=3)  

# Create a sample of 20 loan income data from the original dataset
sample20 = resample(loans_income, n_samples=20, replace=False)

results = []
# Perform the resampling process 2000 times
for nrepeat in range(2000):
    # Resample from the 20 samples with replacement and 
    # compute the sample mean, and append it to the results
    sample = resample(sample20, n_samples=len(sample20), replace=True)
    results.append(sample.mean())  

# Convert the results list into a Pandas Series for easier analysis
results = pd.Series(results)

# Calculate the 5th and 95th percentiles for the 90% confidence interval
confidence_level = 0.9
# The 90% confidence interval corresponds to the middle 90% of the sampling 
# distribution, hence we drop 5% at the two tails. The below code translates to: 
# confidence_interval = list(results.quantile([0.05, 0.95]))
confidence_interval = list(results.quantile([(1-confidence_level)/2, 1-(1-confidence_level)/2]))
print("Confidence interval: ", confidence_interval)

Report also the mean of sample means, which can be used to approximate the population mean.

In [ ]:
print("Mean of 2000 sample means:", results.mean())

----

❓<span style="color:purple;">**Quick test**</span>: Inside the bootstrap loop (the for loop), what would happen if `replace=True` were changed to `replace=False`?

- **A.** The bootstrap procedure would work normally, but run faster.
- **B.** Each resample would contain fewer observations than the original sample.
- **C.** Each resample would contain the same observations as the original sample, only in a different order, so all resampled means would be identical.
- **D.** The 90% confidence interval would become wider.

<details>
<summary>Click to reveal the answer</summary>

**Answer:**

<p style="color:blue;"><b>C. Each resample would contain the same observations as the original sample, only in a different order, so all resampled means would be identical.</b></p>

**Explanation:**

<p style="color:blue;">When changing <code>replace=True</code> to <code>replace=False</code>, the resampling will be performed without replacment. Since the resample size equals the original sample size, every observation in the initial sample will be selected exactly once. As a result, each resample is therefore only a rearrangement of the original sample and has the same mean. </p>
    
<p style="color:blue;">The resulting bootstrap distribution would collapse to a single value, producing no useful estimate of sampling variability and a confidence interval with zero width.</p>

</details>

----

#### **Example 4-2: visualize the 90% confidence interval for the sample mean**

Define a function to visualize the sampling distribution of the statistics (e.g. mean) obtained with bootstrap, as well as the confidence interval.

In [ ]:
def visualize_bootstrap_CI(sample_means, confidence_interval, confidence_level):
    # Create a histogram to visualize the sampling distribution,
    # in this example, it is the sample means from bootstrap
    ax = sample_means.plot.hist(bins=50, figsize=(4, 3))

    # Create a transform where the x-coordinates use data values,
    # while the y-coordinates use proportions of the y-axis height
    axis_transform = ax.get_xaxis_transform()

    # Plot a horizontal black line at 80% of the y-axis height, extending
    # from the lower bound to the upper bound of the confidence interval
    # - confidence_interval: x-coordinates of the points to be plotted,
    #                        extending from the lower bound to the upper bound
    # - [0.8, 0.8]: y-coordinates expressed as proportions of the y-axis height
    ax.plot(confidence_interval, [0.8, 0.8], color='black', transform=axis_transform)

    # Add text annotation to indicate the level of the confidence interval,
    # positioned at 85% of the y-axis height
    ax.text(x=(sum(confidence_interval)/2), y=0.85, s=f'{confidence_level*100:.0f}% interval',
            horizontalalignment='center', verticalalignment='center',
            transform=axis_transform)

    # Draw vertical lines corresponding to the lower and upper bound, and
    # label them with the lower bound value and upper bound value.
    # confidence_interval = [lower_bound_value, upper_bound_value]
    for x in confidence_interval:
        # Draw a vertical line extending from the bottom
        # to 85% of the y-axis height
        ax.axvline(x=x, ymin=0, ymax=0.85, color='black')
        # Add text annotation at 90% of the y-axis height
        ax.text(x, 0.9, f'{x:.0f}',
                horizontalalignment='center', verticalalignment='center',
                transform=axis_transform)

    # Draw a dashed vertical line at the mean of the bootstrap samples,
    # extending from the bottom to 75% of the y-axis height
    meanIncome = sample_means.mean()
    ax.axvline(x=meanIncome, ymin=0, ymax=0.75, color='black', linestyle='--')
    # Add text annotation for the mean value with white background
    # at 10% of the y-axis height
    ax.text(x=meanIncome, y=0.1, s=f'Mean: {meanIncome:.0f}',
            horizontalalignment='center', verticalalignment='center',
            transform=axis_transform,
            # bbox: creates a white background box around the text with a
            # white border and some transparency
            bbox=dict(facecolor='white', edgecolor='white', alpha=0.5))

    # Set the y-axis limit automatically and add 20% of extra space
    # above the plotted histogram
    _, automatic_upper_limit = ax.get_ylim()
    ax.set_ylim(bottom=0, top=automatic_upper_limit * 1.2)
    # Label the y-axis
    ax.set_ylabel('Counts')

    # Adjust the layout to prevent overlap of elements
    plt.tight_layout()

    # Display the plot
    plt.show()

Call the function above to visualize the 90% confidence interval of the sample mean

In [ ]:
visualize_bootstrap_CI(results, confidence_interval, confidence_level=0.9)

The above plot shows the sampling distribution of the 2000 sample means obtained through bootstrapping. It also highlights the 90% confidence interval. The dashed vertical line corresponds to the mean of the 2000 sample means. 

Pay attention, **the confidence interval may vary if a different set of 2000 sample means are obtained**, due to the random sampling variability. This can be confirmed by commenting out the np.random.seed(seed=3) line of code in Example 4-1 and rerun the code.

---

# 🏠 Assignment questions (20 points)

### <span style="color:blue">[5 points]</span> Question-1: implement the equal stratified sampling and compare the result against the proportional stratified sampling.

**Task:** Draw a sample of the same total size (1000) using **equal** stratified sampling and compute the sample mean. Compare the results against the one obtained using proportional stratified sampling.


#### Question 1-1: Determine the number of samples to be taken from each stratum for equal stratified sampling. And display it as a separate column in the income_table. 

Note: for the remainder sample, you can take it from any group.

In [ ]:
# insert your code here

#### Question 1-2: Conduct the equal stratified sampling and estimate the mean income. Print out the population mean, sample mean for equal stratified sampling, as well as the absolution error. 

In [ ]:
# insert your code here

#### Question 1-3: Add an extra row in the summary table (the one in Example 2) corresponding to equal stratified sampling and display it

In [ ]:
# insert your code here

#### Question 1-4: Based on the results from Q1-3, answer which provides a closer estimation of the mean income: proportional stratified sampling or equal stratified sampling? Explain your answer and support your answer with numeric values.

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### <span style="color:blue">[5 Points]</span> Question-2: plot and compare the sampling distribution for different sample sizes (10 and 90) as well as their standard errors. Hint: you can refer to Example 3 for this exercise.


#### Question 2-1: repeatedly draw 1,000 random samples for each sample size (10, 90) and calculate the mean income of each sample. Then plot the resulting sampling distributions, and their empirical standard errors.

In [ ]:
# insert your code here

#### Question 2-2: based on the above plot, answer how does the empirical standard error change when sample size is increased from 10 to 90? Support your answer with numerical evidence. Is the change consistent with the theoretical standard error calculation? Explain your answer.

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----


### <span style="color:blue">[5 Points]</span> Question-3: visualize the 95% confidence interval of the sample means obtained through bootstrap. Each sample should contain 30 values. Hint: refer to Example 4 for the number of samples needed for 95% confidence interval.

#### Question 3-1: compute the 95% confidence interval for the sample mean. Print the mean of sample means and the 95% confidence interval.

In [ ]:
# insert your code here

#### Question 3-2: visualize the 95% confidence interval for the sample means

In [ ]:
# insert your code here

### <span style="color:blue"> [5 points] </span> Question-4: Bootstrap the 95% confidence interval for proportion of loan applicants that have income larger than a given threshold. Each sample should contain 50 values. 

As stated in the lecture slide #14, bootstrap can be used to estimate a confidence interval (CI) for many statistics-not just the mean: medians, proportions, regression coefficients, etc. In this exercise, you need to compute the confidence interval for the average proportion of loan applicants that have income larger than a given threshold. 

**Hint**: Draw a simple random sample of size 50 from `loans_income`. And then use bootstrap with the appropriate number of replicates to estimate the 95% CI for the proportion. 

#### Define the threshold for the income. 

In [ ]:
threshold = loans_income.median()

# Compute the proportion of loan applicants have income above median.
# np.mean(loans_income > threshold) converts True to 1, and False to 0. 
# The mean value gives the proportion above the threshold. 
prop_above_median_population = np.mean(loans_income > threshold)
prop_above_median_population

Here we will use the **median** salary of the whole dataset as the threshold. By definition, it means that half of the loan applicants have an income lower than or equal to the value, half above. 

In the above code, we compute explicitly the proportion of loan applicants have an income larger than this one. As shown in the result, this represents 49.378% of the whole dataset. It is slightly less than 50% could be because values equal to the median are not counted as "> median".

#### Question 4-1: compute the 95% confidence interval for the proportion above threshold using bootstrapping 

In [ ]:
# insert your code here

#### Question 4-2: print the mean of the sample proportions, which can be used to approximate the population proportion. 

In [ ]:
# insert your code here

#### Question 4-3: does the 95% confidence interval computed using bootstrap successfully capture the true population proportion? Explain your answer and support your answer with numeric values. 

----
<div style="color:blue">
    
**Answer**: insert your answer here


</div>

----
